In [14]:
# Card & Krueger (1994) Replication
# Phase 2: Deliverables & Submission Instructions

import pandas as pd
import numpy as np

col_names = [
    'sheet', 'chain', 'co_owned', 'state',
    'southj', 'centralj', 'northj', 'pa1', 'pa2', 'shore',
    'ncalls', 'empft', 'emppt', 'nmgrs', 'wage_st',
    'inctime', 'firstinc', 'bonus', 'pctaff', 'meals',
    'open', 'hrsopen', 'psoda', 'pfry', 'pentree',
    'nregs', 'nregs11',
    'type2', 'status2', 'date2', 'ncalls2',
    'empft2', 'emppt2', 'nmgrs2', 'wage_st2',
    'inctime2', 'firstinc2', 'special2', 'meals2',
    'open2r', 'hrsopen2', 'psoda2', 'pfry2', 'pentree2',
    'nregs2', 'nregs112'
]

DATA_URL = "https://raw.githubusercontent.com/Louisa328/DA-Midterm-Project-Minimum-Wages-and-Employment/main/data/raw/public.dat"
df = pd.read_csv(DATA_URL, sep=r'\s+', names=col_names, na_values='.')

print(f"Shape: {df.shape}")
print(f"NJ restaurants: {(df['state']==1).sum()}")
print(f"PA restaurants: {(df['state']==0).sum()}")
df.head()

# Construct FTE (Full-Time Equivalent)
df['fte'] = df['empft'] + df['nmgrs'] + 0.5 * df['emppt']
df['fte2'] = df['empft2'] + df['nmgrs2'] + 0.5 * df['emppt2']

print(df[['fte', 'fte2']].describe())

Shape: (410, 46)
NJ restaurants: 331
PA restaurants: 79
              fte        fte2
count  398.000000  396.000000
mean    20.998869   21.054293
std      9.749805    9.094453
min      5.000000    0.000000
25%     14.562500   14.500000
50%     19.500000   20.500000
75%     24.500000   26.500000
max     85.000000   60.500000


In [15]:
# Descriptive Statistics
# Group by state: 1=NJ (treatment), 0=PA (control)

nj = df[df['state'] == 1]
pa = df[df['state'] == 0]

print("=== FTE Employment: Mean (Std Dev) ===")
print(f"NJ Before: {nj['fte'].mean():.2f} ({nj['fte'].std():.2f})")
print(f"NJ After:  {nj['fte2'].mean():.2f} ({nj['fte2'].std():.2f})")
print(f"PA Before: {pa['fte'].mean():.2f} ({pa['fte'].std():.2f})")
print(f"PA After:  {pa['fte2'].mean():.2f} ({pa['fte2'].std():.2f})")

=== FTE Employment: Mean (Std Dev) ===
NJ Before: 20.44 (9.11)
NJ After:  21.03 (9.29)
PA Before: 23.33 (11.86)
PA After:  21.17 (8.28)


In [16]:
# Manual DID Calculation
NJ_before = nj['fte'].mean()
NJ_after  = nj['fte2'].mean()
PA_before = pa['fte'].mean()
PA_after  = pa['fte2'].mean()

diff_NJ = NJ_after - NJ_before
diff_PA = PA_after - PA_before
DID     = diff_NJ - diff_PA

print(f"NJ change (after - before): {diff_NJ:.2f}")
print(f"PA change (after - before): {diff_PA:.2f}")
print(f"DID estimate:               {DID:.2f}")

NJ change (after - before): 0.59
PA change (after - before): -2.17
DID estimate:               2.75


In [17]:
# DID Regression
import statsmodels.formula.api as smf

# Convert wide to long format
before = df[['sheet', 'state', 'fte']].copy()
before['post'] = 0
before['employment'] = before['fte']

after = df[['sheet', 'state', 'fte2']].copy()
after['post'] = 1
after['employment'] = after['fte2']

df_long = pd.concat([before, after], ignore_index=True)
df_long['treat'] = df_long['state']  # 1=NJ, 0=PA

print(df_long.shape)
df_long.head()

(820, 7)


,sheet,state,fte,post,employment,fte2,treat
0,46,0,40.50,0,40.50,NaN,0
1,49,0,13.75,0,13.75,NaN,0
2,506,0,8.50,0,8.50,NaN,0
3,56,0,34.00,0,34.00,NaN,0
4,61,0,24.00,0,24.00,NaN,0


In [18]:
# Drop rows with missing employment
df_long = df_long.dropna(subset=['employment'])

# DID Regression with clustered standard errors
model = smf.ols('employment ~ treat + post + treat:post', data=df_long)
results = model.fit(cov_type='cluster', cov_kwds={'groups': df_long['sheet']})

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             employment   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.806
Date:                Fri, 27 Feb 2026   Prob (F-statistic):              0.146
Time:                        22:15:43   Log-Likelihood:                -2904.2
No. Observations:                 794   AIC:                             5816.
Df Residuals:                     790   BIC:                             5835.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.347     17.327      0.0

In [19]:
# Summary of Results
print("=== Replication Results Summary ===")
print(f"Manual DID estimate:      {DID:.4f}")
print(f"Regression DID estimate:  {results.params['treat:post']:.4f}")
print(f"P-value:                  {results.pvalues['treat:post']:.4f}")
print("\nConclusion: The minimum wage increase did NOT reduce employment.")
print("NJ restaurants increased FTE by ~2.75 relative to PA (p<0.05).")

=== Replication Results Summary ===
Manual DID estimate:      2.7536
Regression DID estimate:  2.7536
P-value:                  0.0351

Conclusion: The minimum wage increase did NOT reduce employment.
NJ restaurants increased FTE by ~2.75 relative to PA (p<0.05).
